# 02 · Model A: LightGBM on 31 lexical features

This notebook trains Model A, a LightGBM on 31 numbers computed from each URL, and two LightGBM models kept for comparison. Each is tuned inside the fitting data, then calibrated and given its thresholds on validation, and saved. No test set is opened here.

| Model file | Trained on | Role |
|---|---|---|
| `lgbm.txt` | PhreshPhish fitting data, **main track** (query string removed) | **Model A**, deployed in the app |
| `lgbm_full_urls.txt` | The same rows, **full URLs** (query string kept) | Labelled comparison: how much does the query-string artefact inflate scores? Never deployed |
| `lgbm_phiusiil.txt` | PhiUSIIL's own training split | Cautionary baseline: the benchmark trap |

**Why LightGBM.** Gradient-boosted trees are the standard strong baseline for tabular features. Each one tunes and trains in two to five minutes on CPU, scores a URL in well under a millisecond, need no feature scaling, and every prediction can be broken down into per-feature contributions, which the app shows to the user.

**Pitfalls.** The model sees only the 31 numbers computed from the string. It cannot know anything those numbers do not capture, it is brittle when the data it meets differs from its training data, and it will exploit any feature that separates the classes for the wrong reason. Notebook 01 removed the one such feature the audit found (the query string) and the comparison model below measures what it would have been worth.

**Rules followed here.** Tuning uses cross-validation inside the fitting data only. The validation set (the most recent days of the train split) is used once per model, to fit the calibrator and choose thresholds. No evaluation set is opened in this notebook.

In [1]:
import sys
import warnings

# Progress bars need ipywidgets, which this environment does not install; the warning is noise.
warnings.filterwarnings("ignore", message="IProgress not found")
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src import data as D
from src import features as F
from src import models as M

pd.set_option("display.width", 200, "display.float_format", "{:.4f}".format)
D.set_seeds(D.SEED)

fit = D.load_processed("phreshphish_train")
val = D.load_processed("phreshphish_val")
print(f"Fitting data: {len(fit):,} URLs ({fit['label'].mean():.1%} phishing), {fit['date'].min().date()} to {fit['date'].max().date()}")
print(f"Validation:   {len(val):,} URLs ({val['label'].mean():.1%} phishing), {val['date'].min().date()} to {val['date'].max().date()}")

Fitting data: 418,143 URLs (47.6% phishing), 2024-07-02 to 2025-07-28
Validation:   76,009 URLs (25.0% phishing), 2025-07-29 to 2025-09-07


## 1. The features

`src/features.py` computes 31 numbers from each URL. The list was written before any PhreshPhish row was examined and has been frozen since. In the main track the query string is removed first, so `query_len` and `n_query_params` are always 0.

In [2]:
example = "http://paypal.com.account-check.example/verify/login.php?session=8f2a#top"     # an invented URL
pd.DataFrame({"full URL": F.extract_features(example), "main track": F.extract_features(example, strip_query=True)}).drop("tld").T

,url_len,host_len,path_len,query_len,n_digits,n_letters,n_special,digit_ratio,letter_ratio,special_ratio,...,tld_is_common,tld_is_cc,host_is_ip,has_at,has_double_slash,has_port,has_punycode,is_https,n_brand_words,n_cred_words
full URL,73,32,17,12,2,58,13,0.0274,0.7945,0.1781,...,0,0,0,0,0,0,0,0,1,2
main track,56,32,17,0,0,46,10,0.0000,0.8214,0.1786,...,0,0,0,0,0,0,0,0,1,2


## 2. Training procedure

The same procedure is used for all three models (`src.models.train_lgbm_model`):

1. **Tune** with Optuna: 20 trials, each scored by 3-fold stratified cross-validation on a 100,000-row stratified sample of the fitting data. The objective is the mean PR-AUC across folds. Six settings are tuned: learning rate, number of leaves, minimum rows per leaf, feature and row sampling, and L2 regularization. Each fold uses early stopping on its own held-out part.
2. **Class weights:** phishing rows are weighted by (legitimate rows / phishing rows), so both classes count equally.
3. **Fit** on all fitting rows with the best settings. The number of boosting rounds is the mean best round of the winning trial.
4. **Calibrate and set thresholds on validation only** (section 4).

Seeds are fixed (Python, NumPy, Optuna and LightGBM, with LightGBM's deterministic mode on), so a rerun reproduces the same model.

### Model A: main track

In [3]:
main = M.train_lgbm_model("lgbm", "main", fit, val)
print(f"Trained in {main['seconds']:.0f} s")
main["trials"].sort_values("cv_pr_auc", ascending=False).head(8)

Trained in 290 s


,trial,cv_pr_auc,mean_rounds,learning_rate,num_leaves,min_child_samples,feature_fraction,bagging_fraction,lambda_l2
14,14,0.9875,327.3333,0.0289,225,11,0.7785,0.8752,0.0011
13,13,0.9874,553.3333,0.0207,167,12,0.7854,0.7820,0.0014
16,16,0.9874,377.6667,0.0350,155,33,0.7595,0.9277,0.0012
12,12,0.9873,636.3333,0.0215,131,11,0.8335,0.7838,0.0061
15,15,0.9873,523.0000,0.0212,179,26,0.7525,0.7766,0.0054
11,11,0.9873,434.6667,0.0225,192,18,0.9062,0.7928,0.0033
19,19,0.9873,258.6667,0.0384,230,21,0.6983,0.7965,0.0074
7,7,0.9873,446.3333,0.0216,197,21,0.8650,0.7247,0.1203


### Comparison model: the same rows with the query string kept

This model differs from Model A in one thing only: it sees the URL as shipped, including the query string that the collector removed from legitimate URLs but not from phishing ones.

In [4]:
full = M.train_lgbm_model("lgbm_full_urls", "full", fit, val)
print(f"Trained in {full['seconds']:.0f} s")

Trained in 278 s


### Cautionary baseline: LightGBM trained on PhiUSIIL

Trained and calibrated entirely inside PhiUSIIL (its own 70% training split and 15% validation split), the way the benchmark is normally used. Every legitimate PhiUSIIL URL is `https://www.<host>/`, so a near-perfect score here is expected and says nothing about phishing.

In [5]:
phi_fit, phi_val = D.load_processed("phiusiil_train"), D.load_processed("phiusiil_val")
baseline = M.train_lgbm_model("lgbm_phiusiil", "main", phi_fit, phi_val)
print(f"Trained in {baseline['seconds']:.0f} s on {len(phi_fit):,} PhiUSIIL URLs")

Trained in 121 s on 162,884 PhiUSIIL URLs


## 3. Validation results

Read back from the saved `models/*_operating.json` files. These are validation numbers, used here only to check that training worked. The test results are in notebook 04.

In [6]:
saved = {name: M.load_json(M.MODELS_DIR / f"{name}_operating.json") for name in ("lgbm", "lgbm_full_urls", "lgbm_phiusiil")}
table = pd.DataFrame({
    name: {
        "validation set": "PhiUSIIL validation" if name == "lgbm_phiusiil" else "PhreshPhish validation",
        "fitting rows": f"{op['training']['fit_rows']:,}",
        "boosting rounds": op["training"]["n_rounds"],
        "cross-validated PR-AUC": f"{op['training']['tuning']['best_cv_pr_auc']:.4f}",
        "validation ROC-AUC": f"{op['training']['validation_roc_auc']:.4f}",
        "validation PR-AUC": f"{op['training']['validation_pr_auc']:.4f}",
        "detection at the block threshold (0.1% FPR)": f"{op['thresholds']['block']['validation_tpr']:.1%}",
        "false-positive rate at the warn threshold (95% detection)": f"{op['thresholds']['warn']['validation_fpr']:.2%}",
    } for name, op in saved.items()
})
table

,lgbm,lgbm_full_urls,lgbm_phiusiil
validation set,PhreshPhish validation,PhreshPhish validation,PhiUSIIL validation
fitting rows,"418,143","418,143","162,884"
boosting rounds,327,349,352
cross-validated PR-AUC,0.9875,0.9888,0.9946
validation ROC-AUC,0.8901,0.8922,0.9944
validation PR-AUC,0.8342,0.8464,0.9947
detection at the block threshold (0.1% FPR),16.3%,25.4%,93.2%
false-positive rate at the warn threshold (95% detection),65.92%,65.89%,0.40%


In [7]:
a, c, b = (saved[k]["training"] for k in ("lgbm", "lgbm_full_urls", "lgbm_phiusiil"))
too_good = [name for name, op in saved.items() if op["training"]["validation_roc_auc"] > 0.999]
display(Markdown(
    f"- **Model A** reaches a validation ROC-AUC of **{a['validation_roc_auc']:.4f}**. Its cross-validated PR-AUC inside the "
    f"fitting data was {a['tuning']['best_cv_pr_auc']:.4f}; validation is later in time, so a lower number there is expected.\n"
    f"- **The comparison model on full URLs** reaches {c['validation_roc_auc']:.4f} on the same validation rows. "
    f"The difference, {c['validation_roc_auc'] - a['validation_roc_auc']:+.4f} ROC-AUC, is what the query-string artefact adds on this data.\n"
    f"- **The PhiUSIIL baseline** reaches {b['validation_roc_auc']:.4f} on PhiUSIIL's own validation split.\n"
    f"- Models with validation ROC-AUC above 0.999, the project's flag for a possible artefact: **{too_good or 'none'}**."
))

- **Model A** reaches a validation ROC-AUC of **0.8901**. Its cross-validated PR-AUC inside the fitting data was 0.9875; validation is later in time, so a lower number there is expected.
- **The comparison model on full URLs** reaches 0.8922 on the same validation rows. The difference, +0.0021 ROC-AUC, is what the query-string artefact adds on this data.
- **The PhiUSIIL baseline** reaches 0.9944 on PhiUSIIL's own validation split.
- Models with validation ROC-AUC above 0.999, the project's flag for a possible artefact: **none**.

The project's rule is that a ROC-AUC above 0.999 is treated as a possible artefact and its cause is found before going on. No model reaches that flag on validation. The PhiUSIIL baseline comes closest (0.9944 on PhiUSIIL's own validation split, in the output above), and its cause is already known: notebook 01 showed that one template describes every legitimate PhiUSIIL URL. That is why it is only a baseline.

### Which features does Model A use?

Gain is the total improvement in the training objective from splits on a feature. It shows what the model relies on, not whether that reliance is justified.

In [8]:
gain = pd.DataFrame({
    "Model A (main track)": pd.Series(main["booster"].feature_importance("gain"), index=F.FEATURE_NAMES),
    "Comparison (full URLs)": pd.Series(full["booster"].feature_importance("gain"), index=F.FEATURE_NAMES),
    "PhiUSIIL baseline": pd.Series(baseline["booster"].feature_importance("gain"), index=F.FEATURE_NAMES),
})
share = (gain / gain.sum()).sort_values("Model A (main track)", ascending=False)
display(share.head(12).style.format("{:.1%}").set_caption("Share of total gain, top 12 features of Model A"))
print("Query features, share of gain:")
print(share.loc[["query_len", "n_query_params"]].sum().map("{:.2%}".format).to_string())

,Model A (main track),Comparison (full URLs),PhiUSIIL baseline
path_len,32.2%,32.7%,8.4%
host_entropy,7.3%,7.1%,1.7%
tld_is_common,6.9%,7.1%,1.5%
path_depth,6.7%,6.2%,2.5%
url_entropy,6.6%,10.4%,0.6%
special_ratio,6.5%,3.4%,0.3%
n_host_hyphens,5.5%,4.4%,0.2%
host_len,4.3%,4.2%,0.8%
is_https,4.0%,4.3%,35.7%
subdomain_depth,2.4%,2.2%,4.1%


Query features, share of gain:
Model A (main track)      0.00%
Comparison (full URLs)    1.57%
PhiUSIIL baseline         0.00%


## 4. Calibration and the two decision thresholds

Both are fitted on the validation set only and saved in `models/lgbm_operating.json`.

- **Raw score:** the model's log-odds output. Thresholds are set on it.
- **Calibrated score:** an isotonic (monotone, step-wise) map from raw score to a number between 0 and 1. It is stored as a list of breakpoints and applied with `numpy.interp`, so the app does not need scikit-learn.
- **Block threshold:** the lowest raw score at which no more than 0.1% of legitimate validation URLs are flagged.
- **Warn threshold:** the highest raw score at which at least 95% of phishing validation URLs are flagged.

A URL at or above the block threshold is blocked; between the two it gets a warning; below the warn threshold it is allowed.

**A calibrated score is not a real-world probability.** Validation is 25% phishing because of when the collector gathered each class, and real traffic is far below that. The thresholds do not have this problem: they are set on rates within each class.

In [9]:
rows = {}
for name, op in saved.items():
    t = op["thresholds"]
    rows[name] = {
        "block: raw score": t["block"]["raw_score"], "block: calibrated": t["block"]["calibrated_score"],
        "block: validation FPR": t["block"]["validation_fpr"], "block: validation detection": t["block"]["validation_tpr"],
        "block: false positives": t["block"]["validation_false_positives"],
        "warn: raw score": t["warn"]["raw_score"], "warn: calibrated": t["warn"]["calibrated_score"],
        "warn: validation FPR": t["warn"]["validation_fpr"], "warn: validation detection": t["warn"]["validation_tpr"],
        "legitimate validation rows": t["validation_rows"]["legitimate"],
    }
pd.DataFrame(rows).T.style.format({
    "block: raw score": "{:.3f}", "block: calibrated": "{:.4f}", "block: validation FPR": "{:.3%}",
    "block: validation detection": "{:.1%}", "block: false positives": "{:.0f}", "warn: raw score": "{:.3f}",
    "warn: calibrated": "{:.4f}", "warn: validation FPR": "{:.2%}", "warn: validation detection": "{:.1%}",
    "legitimate validation rows": "{:,.0f}"})

,block: raw score,block: calibrated,block: validation FPR,block: validation detection,block: false positives,warn: raw score,warn: calibrated,warn: validation FPR,warn: validation detection,legitimate validation rows
lgbm,4.434,0.9731,0.100%,16.3%,57,-5.422,0.0817,65.92%,95.0%,"57,001"
lgbm_full_urls,4.419,0.9704,0.100%,25.4%,57,-5.517,0.0649,65.89%,95.0%,"57,001"
lgbm_phiusiil,2.425,0.8366,0.099%,93.2%,20,1.288,0.7979,0.40%,95.0%,"20,153"


The block threshold rests on a few dozen false positives among the legitimate validation rows, so the false-positive rate it achieves on new data will vary around 0.1%. Notebook 04 reports what it actually is on each test set, with a confidence interval.

## 5. Saved files and hashes

In [10]:
files = [M.MODELS_DIR / f for f in ("lgbm.txt", "lgbm_operating.json", "lgbm_full_urls.txt", "lgbm_full_urls_operating.json",
                                    "lgbm_phiusiil.txt", "lgbm_phiusiil_operating.json")]
hashes = M.write_hashes(files + [ROOT / "src" / "features.py"])
pd.DataFrame(hashes).T.loc[[f"models/{f.name}" for f in files] + ["src/features.py"]]

,sha256,bytes
models/lgbm.txt,6ef2b4b710b5834e5453281d5751f91eb124fa2dbd308f...,8031679
models/lgbm_operating.json,b8a5f87bf762df64b496621c984ba06610bdc6e9c91966...,10424
models/lgbm_full_urls.txt,10fccf36343c9faac8e5fcb7e9b9bb7487a9ae93dabc11...,9348964
models/lgbm_full_urls_operating.json,05e62db8553bc598d3cdf045a8fae20da2b9a17891828f...,10270
models/lgbm_phiusiil.txt,08ee26e8bfbf4b64ce860729bfd8c1a4c844935c4e2689...,744677
models/lgbm_phiusiil_operating.json,63b1279b6012fb757a76e5cc4e640c9da4ed99cf71c280...,8111
src/features.py,1d9f941f40432f0acd7dd4cbafd9fa08865110bef14c5b...,20209


`models/hashes.json` records the SHA-256 of each model file and of `src/features.py`, so the deployed app can be checked against exactly what was trained here.

## 6. Conclusion

In [11]:
a, b = (M.load_json(M.MODELS_DIR / f"{name}_operating.json") for name in ("lgbm", "lgbm_phiusiil"))
display(Markdown(
    f"- **Model A is trained and frozen.** On validation, the six weeks after the fitting data, its PR-AUC is "
    f"**{a['training']['validation_pr_auc']:.3f}**, lower than the {a['training']['tuning']['best_cv_pr_auc']:.3f} "
    "that cross-validation inside the fitting data suggested "
    f"(validation ROC-AUC {a['training']['validation_roc_auc']:.3f}).\n"
    f"- **The block threshold** flags {a['thresholds']['block']['validation_fpr']:.1%} of legitimate validation URLs and "
    f"catches {a['thresholds']['block']['validation_tpr']:.1%} of phishing.\n"
    f"- **The warn threshold** catches {a['thresholds']['warn']['validation_tpr']:.0%} of phishing but flags "
    f"{a['thresholds']['warn']['validation_fpr']:.1%} of legitimate URLs: already too wide to act on before any test.\n"
    f"- **The PhiUSIIL baseline** looks excellent on its own data (validation ROC-AUC "
    f"{b['training']['validation_roc_auc']:.4f}), which notebook 04 shows does not carry over.\n\n"
    "Next: `03_cnn.ipynb` trains Model B on the same fitting data."
))

- **Model A is trained and frozen.** On validation, the six weeks after the fitting data, its PR-AUC is **0.834**, lower than the 0.987 that cross-validation inside the fitting data suggested (validation ROC-AUC 0.890).
- **The block threshold** flags 0.1% of legitimate validation URLs and catches 16.3% of phishing.
- **The warn threshold** catches 95% of phishing but flags 65.9% of legitimate URLs: already too wide to act on before any test.
- **The PhiUSIIL baseline** looks excellent on its own data (validation ROC-AUC 0.9944), which notebook 04 shows does not carry over.

Next: `03_cnn.ipynb` trains Model B on the same fitting data.